# rl_report

> Phase 12, Track B report: the `RL_PLAN` analyses of the evaluation rows (main boards, gains over the warm-start model, reward schemes, Phase 11 Haiku and rule agents, information use, generalisation, follow share, cross-play, liar, per-role adapters), `report.md` with provenance first, and figures. All findings are exploratory.

In [ ]:
#| default_exp rl_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.llm_figures import _save
from explRL.llm_report import _f, _fmt, _paired, _sel, _stat

INFO = ('sharePerHeld', 'redundantShares', 'shareToUse', 'guesses', 'soloShare', 'wrongPerSolve')
SCHEMES = ('individual', 'team', 'credit')


def _ev(rows, model, test, **kw):
    return _sel(rows, study='coopEval', level=f'{model}|{test}', **kw)


def _seeds(rows, seeds):
    return [r for r in rows if int(_f(r['seed'])) in seeds]


def rlSummaries(rows, p11, numpyRows=()):
    "RL_PLAN: main, gains, schemes, against Phase 11, information, generalisation, follow share, cross-play, liar, per-role."
    ev = _sel(rows, study='coopEval')
    models = sorted({r['level'].split('|')[0] for r in ev}, key=lambda m: (('untrained', 'warm') + SCHEMES + ('perRole',)).index(m)
                    if m in ('untrained', 'warm') + SCHEMES + ('perRole',) else 99)
    haiku = lambda ch, N: _seeds(_sel(p11, study='clue', runner='llm', channel=ch, N=N), range(900, 905))
    rule = _seeds(_sel(list(numpyRows), study='clueNumpy', channel='blackboard', N=4), range(900, 905))
    S = {k: {} for k in ('main', 'info', 'gain', 'schemes', 'phase11', 'gen', 'follow', 'cross', 'liar', 'perRole')}
    for m in models:
        main = _ev(rows, m, 'main')
        S['main'][m] = {k: _stat(_f(r[k]) for r in main) for k in ('welfare', 'pooledShare')}
        S['info'][m] = {k: _stat(_f(r[k]) for r in main) for k in INFO}
        S['phase11'][f'{m} - Haiku'] = _stat(_paired(_seeds(main, range(900, 905)), haiku('blackboard', 4), 'welfare'))
        if rule:
            S['phase11'][f'{m} - rule agents'] = _stat(_paired(_seeds(main, range(900, 905)), rule, 'welfare'))
        if m in SCHEMES:
            S['gain'][m] = _stat(_paired(main, _ev(rows, 'warm', 'main'), 'welfare'))
        for t, ch, N in (('gen8', 'blackboard', 8), ('genMsg', 'messages', 4), ('genLead', 'hierarchical', 4)):
            if _ev(rows, m, t):
                S['gen'][f'{m} {t} - Haiku'] = _stat(_paired(_ev(rows, m, t), haiku(ch, N), 'welfare'))
        if _ev(rows, m, 'genLead'):
            S['follow'][m] = _stat(_f(r['followShare']) for r in _ev(rows, m, 'genLead'))
        for t in ('cross1', 'cross3'):
            if _ev(rows, m, t):
                S['cross'][f'{m} {t} - all Haiku'] = _stat(_paired(_ev(rows, m, t), haiku('blackboard', 4), 'welfare'))
        if _ev(rows, m, 'liar'):
            loss = {}
            for ch, clean in (('blackboard', _ev(rows, m, 'main')), ('messages', _ev(rows, m, 'genMsg'))):
                c = {int(_f(r['seed'])): _f(r['welfareHonest']) for r in clean}
                lr = {int(_f(r['seed'])): _f(r['welfareHonest']) for r in _ev(rows, m, 'liar', channel=ch)}
                loss[ch] = {s: c[s] - lr[s] for s in sorted(set(c) & set(lr))}
                S['liar'][f'{m} honest loss, {ch}'] = _stat(loss[ch].values())
            both = sorted(set(loss['blackboard']) & set(loss['messages']))
            S['liar'][f'{m} blackboard - messages loss (H3: > 0)'] = _stat(loss['blackboard'][s] - loss['messages'][s] for s in both)
    for a, b in (('credit', 'individual'), ('team', 'individual'), ('credit', 'team')):
        if a in models and b in models:
            S['schemes'][f'{a} - {b}'] = _stat(_paired(_ev(rows, a, 'main'), _ev(rows, b, 'main'), 'welfare'))
    if 'perRole' in models and 'credit' in models:
        S['perRole']['per-role - shared (credit)'] = _stat(_paired(_ev(rows, 'perRole', 'main'), _ev(rows, 'credit', 'main'), 'welfare'))
    return S


def writeRL(path, S, meta):
    "report.md for Track B: provenance first, then the RL_PLAN tables (exploratory)."
    L = [f"# Phase 12, Track B: training agents to work together ({meta['mode']})", '', 'Exploratory, pre-specified in `RL_PLAN`.', '',
         f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; base model {meta['base']}; adapters "
         + ', '.join(f'{k} `{v[:12]}`' for k, v in meta['adapters'].items()),
         f"- GPU time {meta['gpuHours']:.1f} h, GPU cost ${meta['gpuCost']:.2f} (cap $150); API cost of this evaluation "
         f"${meta['apiCost']:.2f}; total API spend ${meta['apiSpent']:.2f} of the $150 cap", '',
         '## Main boards (N = 4, blackboard, 40 turns)', '', '| model | welfare per agent-turn | pooled-bound share |', '|---|---|---|']
    L += [f"| {m} | {_fmt(v['welfare'])} | {_fmt(v['pooledShare'])} |" for m, v in S['main'].items()]
    L += ['', '## Information use (means)', '', '| model | ' + ' | '.join(INFO) + ' |', '|---' * (len(INFO) + 1) + '|']
    L += [f'| {m} | ' + ' | '.join(f"{v[k]['mean']:.2f}" for k in INFO) + ' |' for m, v in S['info'].items()]
    for title, key in (('RL gain over the warm-start model', 'gain'), ('Reward schemes', 'schemes'),
                       ('Against Phase 11 (seeds 900-904)', 'phase11'), ('Generalisation', 'gen'), ('Follow share (coordinator)', 'follow'),
                       ('Cross-play with Haiku', 'cross'), ('Liar', 'liar'), ('Per-role adapters (optional)', 'perRole')):
        if S[key]:
            L += ['', f'## {title}', '', '| contrast | mean [95% CI] | n |', '|---|---|---|']
            L += [f"| {k} | {_fmt(v)} | {v['n']} |" for k, v in S[key].items()]
    path.write_text('\n'.join(L) + '\n')
    return path


def _bars(ax, items, title, ref=()):
    m = np.array([v['mean'] for _, v in items])
    err = [m - np.array([v['ci'][0] for _, v in items]), np.array([v['ci'][1] for _, v in items]) - m]
    ax.errorbar(m, range(len(items)), xerr=err, fmt='o', capsize=3)
    ax.set_yticks(range(len(items)), [k for k, _ in items], fontsize=8)
    for x, lab, c in ref:
        ax.axvline(x, color=c, ls='--', lw=1, label=lab)
    ax.invert_yaxis()
    ax.set(title=title)


def writeRLFigures(out, S, curves=None, refs=()):
    "coop_main.png, coop_info.png, coop_gen.png and (with training curves {scheme: metrics rows}) coop_curves.png."
    fig, ax = plt.subplots(figsize=(7, 3.5))
    _bars(ax, [(m, v['welfare']) for m, v in S['main'].items()], 'Main boards: welfare per agent-turn', refs)
    if refs:
        ax.legend(fontsize=8)
    paths = [_save(fig, out / 'coop_main.png')]
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))
    for ax, k in zip(axs, ('sharePerHeld', 'soloShare', 'wrongPerSolve')):
        _bars(ax, [(m, v[k]) for m, v in S['info'].items()], k)
    paths.append(_save(fig, out / 'coop_info.png'))
    items = [(k, v) for key in ('gen', 'cross', 'liar') for k, v in S[key].items()]
    if items:
        fig, ax = plt.subplots(figsize=(9, 0.32 * len(items) + 1.2))
        _bars(ax, items, 'Generalisation, cross-play and liar (differences)', [(0, None, 'k')])
        paths.append(_save(fig, out / 'coop_gen.png'))
    if curves:
        fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
        for sc, rs in curves.items():
            for ax, k in zip(axs, ('welfare', 'sharePerHeld')):
                ax.plot([r['step'] for r in rs], [r[k] for r in rs], label=sc)
        for ax, k in zip(axs, ('welfare', 'shares per held clue')):
            ax.set(xlabel='RL step', ylabel=k, title=f'Training: {k}')
        axs[0].legend(fontsize=8)
        paths.append(_save(fig, out / 'coop_curves.png'))
    return paths

## Tests

In [ ]:
import tempfile
from pathlib import Path

# synthetic evaluation rows with known differences (welfare = base + model offset, paired on seed)
OFF = dict(untrained=0.0, warm=0.2, individual=0.25, team=0.3, credit=0.4)
def row(model, test, ch, N, seed, w, **kw):
    r = dict(study='coopEval', level=f'{model}|{test}', channel=ch, N=N, seed=seed, episode=0, runner='local', welfare=w,
             welfareHonest=w, pooledShare=w / 2, followShare=0.5, **{k: 1.0 for k in INFO})
    r.update(kw)
    return r
base = lambda s: 0.1 + 0.01 * (s % 5)
rows = [row(m, 'main', 'blackboard', 4, s, base(s) + o) for m, o in OFF.items() for s in list(range(900, 905)) + list(range(930, 940))]
for m, o in OFF.items():
    if m == 'untrained':
        continue
    rows += [row(m, t, ch, N, s, base(s) + o) for t, ch, N in (('gen8', 'blackboard', 8), ('genMsg', 'messages', 4),
                                                            ('genLead', 'hierarchical', 4), ('cross1', 'blackboard', 4),
                                                            ('cross3', 'blackboard', 4)) for s in range(900, 905)]
    rows += [row(m, 'liar', ch, 4, s, base(s) + o - (0.1 if ch == 'blackboard' else 0.05)) for ch in ('blackboard', 'messages') for s in range(900, 903)]
p11 = [dict(study='clue', runner='llm', channel=ch, N=N, seed=s, episode=0, welfare=base(s) + 0.3)
       for ch, N in (('blackboard', 4), ('blackboard', 8), ('messages', 4), ('hierarchical', 4)) for s in range(900, 905)]
rule = [dict(study='clueNumpy', runner='numpy', channel='blackboard', N=4, seed=s, episode=0, welfare=base(s) + 0.5) for s in range(900, 905)]
S = rlSummaries(rows, p11, rule)
near = lambda v, x: abs(v['mean'] - x) < 1e-9
assert list(S['main']) == list(OFF) and S['main']['credit']['welfare']['n'] == 15
assert near(S['gain']['credit'], 0.2) and near(S['gain']['individual'], 0.05) and 'warm' not in S['gain']
assert near(S['schemes']['credit - individual'], 0.15) and near(S['schemes']['credit - team'], 0.1)
assert near(S['phase11']['credit - Haiku'], 0.1) and near(S['phase11']['credit - rule agents'], -0.1) and S['phase11']['credit - Haiku']['n'] == 5
assert near(S['gen']['warm gen8 - Haiku'], -0.1) and near(S['cross']['team cross1 - all Haiku'], 0.0) and near(S['follow']['credit'], 0.5)
assert near(S['liar']['credit honest loss, blackboard'], 0.1) and near(S['liar']['credit honest loss, messages'], 0.05)
assert near(S['liar']['credit blackboard - messages loss (H3: > 0)'], 0.05) and not S['perRole']
with tempfile.TemporaryDirectory() as d:
    meta = dict(mode='full', planHash='p', commit='c', base='m', adapters=dict(credit='abcdef0123456789'), gpuHours=5.5,
                gpuCost=12.0, apiCost=0.8, apiSpent=48.0)
    text = writeRL(Path(d) / 'report.md', S, meta).read_text()
    assert text.index('`p`') < text.index('## Main boards') and '| credit - individual |' in text and 'Per-role' not in text
    curves = {sc: [dict(step=k, welfare=0.1 * k, sharePerHeld=0.2) for k in (1, 2, 3)] for sc in SCHEMES}
    paths = writeRLFigures(Path(d), S, curves, refs=[(0.4, 'Haiku', 'tab:orange')])
    assert [p.name for p in paths] == ['coop_main.png', 'coop_info.png', 'coop_gen.png', 'coop_curves.png']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()